____
### 1. Imports

In [2]:
import gymnasium as gym
import random
from gymnasium import spaces
import numpy as np
from collections import deque
from collections import defaultdict


____
### 2. Environment

#### 2.1 Classes for Cards and Deck

In [3]:
class Card: # represents a playing card with a suit, colour and number
    VALID_SUITS = {"hearts", "diamonds", "clubs", "spades"} # valid suits for a card
    SUIT_COLOURS = {
        "hearts": "red",
        "diamonds": "red",
        "clubs": "black",
        "spades": "black",
    }

    def __init__(self, suit: str, number: int):
        suit_name = suit.strip().lower()
        if suit_name not in self.VALID_SUITS:
            raise ValueError(f"Invalid suit: {suit}")
        if not 1 <= number <= 13:
            raise ValueError("number must be between 1 and 13")

        self.suit = suit_name
        self.colour = self.SUIT_COLOURS[suit_name] #indexing for suit colours
        self.number = number

    def __repr__(self):
        return f"Card(suit={self.suit!r}, colour={self.colour!r}, number={self.number})"



In [4]:
class Deck:
    def __init__(self, num_decks: int = 1):
        if num_decks < 1:
            raise ValueError("num of decks must be at least 1")

        self.num_decks = num_decks
        self.cards = self.build_deck()

    def build_deck(self):
        suits = ("hearts", "diamonds", "clubs", "spades")
        cards = []

        for _ in range(self.num_decks):
            for suit in suits: #iterating through the number of suits
                for number in range(1, 14): #iterating from 1 to 13 to get the 13 cards
                    cards.append(Card(suit, number))
        return cards

    def shuffle(self): #shuffles the cards in place
        random.shuffle(self.cards)
        return self

    def deal_card(self): 
        if not self.cards:
            raise IndexError("No cards left in the deck")
        return self.cards.pop() #returns the card at the top of the deck

    def __len__(self):
        return len(self.cards)

    def __repr__(self):
        return f"Deck(num_decks={self.num_decks}, cards_remaining={len(self.cards)})"


#### 2.2 Setting up Environment
- Setup: 6-8 decks combined throw away the first 10 cards

Observation Space (7 features):
    - [0] dragon_card_normalized   - Last Dragon card value (1–13 normalized to 0–1)
    - [1] tiger_card_normalized    - Last Tiger card value (1–13 normalized to 0–1)
    - [2] dragon_win_rate          - Rolling Dragon win rate
    - [3] tiger_win_rate           - Rolling Tiger win rate
    - [4] tie_rate                 - Rolling Tie rate
    - [5] bankroll_normalized      - Current bankroll normalized (0–1)
    - [6] rounds_remaining_norm    - Rounds remaining normalized (0–1)

- Rewards:
    - Bet Dragon/Tiger win  → +1.0  (1:1 payout)
    - Bet Dragon/Tiger lose → -1.0
    - Tie when bet Dragon/Tiger → -0.5  (lose half)
    - Bet Tie win          → +11.0  (11:1 payout)
    - Bet Tie lose         → -1.0

Actions:
    0 = Bet Dragon
    1 = Bet Tiger
    2 = Bet Tie
    3 = Skip Round


In [5]:
class DragonTigerEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 1}

    # PAYOUT CONSTANTS
    PAYOUT_DRAGON_TIGER_WIN  =  1.0
    PAYOUT_DRAGON_TIGER_LOSE = -1.0
    PAYOUT_TIE_PUSH          = -0.5   # half bet returned when tie but didn't bet tie
    PAYOUT_TIE_WIN           =  11.0   # 11:1 for correct tie bet
    PAYOUT_TIE_LOSE          = -1.0

    def __init__(
        self,
        num_decks: int = 8, # 6 - 8 decks used 
        initial_bankroll: float = 100.0, # starting amount of money, ends when hit 0
        max_rounds: int = 200, #max rounds per episode before truncation
        bet_size: float = 1.0, #fixed wage per round in units
        history_window: int = 30, #how many past rounds the agent can remember
        render_mode: str = None, #set to "human" to print round by round output
    ):
        super().__init__()

        self.num_decks = num_decks
        self.initial_bankroll = initial_bankroll
        self.max_rounds = max_rounds
        self.bet_size = bet_size
        self.history_window = history_window
        self.render_mode = render_mode

        # Action Space
        self.action_space = spaces.Discrete(5) # 5 possible actions, 
        # 0 = bet on dragon, 1 = bet on tiger, 2 = bet on tie, 3 = skip, 4 = leave

        # Observation Space
        # 7 continuous features, all within [0, 1]
        self.observation_space = spaces.Box( # vectors of 7 numbers, capped between 0 and 1
            low=np.zeros(7, dtype=np.float32),
            high=np.ones(7, dtype=np.float32),
            dtype=np.float32, #numbers are normalised to floats between 0 and 1 to make it easy for the neural network to learn
        ) # 7 numbers represents:
        #[dragon_card, tiger_card, dragon_win_rate, tiger_win_rate, tie_rate, bankroll, rounds_left]

        # Internal state (initialised in reset)
        self.deck: Deck | None = None 
        self.bankroll: float = 0.0
        self.round_num: int = 0
        self.last_dragon_card: int = 0
        self.last_tiger_card: int = 0
        self.outcome_history: deque = deque(maxlen=history_window)  # 0=D, 1=T, 2=Tie

    
# Public API
    def reset(self, seed=None, options=None): # method used to reset the round
        super().reset(seed=seed)

        self.deck = self.build_deck() #
        self.bankroll = self.initial_bankroll #resets the bankroll
        self.round_num = 0 #round num reset to 0
        self.last_dragon_card = 7   # neutral starting card (mid-range) ADJUST THIS LATER
        self.last_tiger_card = 7
        self.outcome_history.clear() # clears outcome history

        obs = self.get_obs()
        info = {"bankroll": self.bankroll, "round": self.round_num}
        return obs, info

    def step(self, action: int, stake: float | None = None): #method to progress the game forward based off the action
        assert self.action_space.contains(action), f"Invalid action: {action}" #if the move is invalid, print invalid
        
        if action == 3:
            stake = 0.0
        else:
            if stake is None:
                stake = self.bet_size
            if stake <= 0:
                raise ValueError("stake must be greater than 0")
        
        dragon_card = self.deal_card() #deal dragon card
        tiger_card  = self.deal_card() #deal tiger card

        self.last_dragon_card = dragon_card.number
        self.last_tiger_card  = tiger_card.number

        if dragon_card.number > tiger_card.number: #determining outcomes
            outcome = 0  # Dragon wins
        elif tiger_card.number > dragon_card.number:
            outcome = 1  # Tiger wins
        else:
            outcome = 2  # Tie

        self.outcome_history.append(outcome)

#ADJUST ACCORDING TO 
        if action == 3:
            reward = 0.0
        else:
            reward = self.calculate_reward(action, outcome) #calculate reward
        self.bankroll += reward * stake
        self.round_num += 1

#TERMINATION CONDITIONS (ADD ONE MORE, when choose to walk away )
        terminated = self.bankroll <= 0 or action == 4
        truncated  = self.round_num >= self.max_rounds

        obs  = self.get_obs()
        info = {
            "bankroll": self.bankroll,
            "round": self.round_num,
            "dragon_card": dragon_card.number,
            "tiger_card": tiger_card.number,
            "outcome": ["Dragon", "Tiger", "Tie"][outcome],
            "action": ["Bet Dragon", "Bet Tiger", "Bet Tie", "Skip Round", "Walk Away"][action],
            "stake": stake,
            "reward": reward,
        }

        if self.render_mode == "human":
            self.render_human(info)

        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode == "human":
            print(f"Round {self.round_num} | Bankroll: ${self.bankroll:.2f}")

    def close(self):
        pass


#PRIVATE HELPER METHODS
    def build_deck(self) -> Deck:
        deck = Deck(self.num_decks)
        deck.shuffle()
        for _ in range(10):
            deck.deal_card()
        return deck

    def deal_card(self) -> Card:
        """Deal one card; rebuild the deck if it is exhausted."""
        if self.deck is None or len(self.deck) == 0:
            self.deck = self.build_deck()
        return self.deck.deal_card()

    def calculate_reward(self, action: int, outcome: int) -> float:
        if outcome == 2:  # Tie
            if action == 2:
                return self.PAYOUT_TIE_WIN
            else:
                return self.PAYOUT_TIE_PUSH  # half lost on Dragon/Tiger bets

        # Non-tie outcomes
        if action == outcome:          # Correct Dragon or Tiger bet
            return self.PAYOUT_DRAGON_TIGER_WIN
        elif action == 2:              # Bet Tie but no tie
            return self.PAYOUT_TIE_LOSE
        else:                          # Wrong side
            return self.PAYOUT_DRAGON_TIGER_LOSE

    def get_obs(self) -> np.ndarray:
        dragon_norm  = (self.last_dragon_card - 1) / 12.0
        tiger_norm   = (self.last_tiger_card  - 1) / 12.0

        history = list(self.outcome_history)
        n = len(history) if history else 1  # avoid /0
        dragon_rate = history.count(0) / n if history else 0.5
        tiger_rate  = history.count(1) / n if history else 0.5
        tie_rate    = history.count(2) / n if history else 0.0

        bankroll_norm  = np.clip(self.bankroll / self.initial_bankroll, 0.0, 1.0)
        rounds_norm    = 1.0 - (self.round_num / self.max_rounds)

        return np.array(
            [dragon_norm, tiger_norm, dragon_rate, tiger_rate, tie_rate,
             bankroll_norm, rounds_norm],
            dtype=np.float32,
        )

    def render_human(self, info: dict):
        card_names = {1: "A", 11: "J", 12: "Q", 13: "K"}
        d = card_names.get(info["dragon_card"], str(info["dragon_card"]))
        t = card_names.get(info["tiger_card"],  str(info["tiger_card"]))
        outcome_icon = {"Dragon": "🐉", "Tiger": "🐯", "Tie": "🤝"}[info["outcome"]]

        print(
            f"Round {info['round']:>3} | "
            f"Dragon: {d:>2}  Tiger: {t:>2}  "
            f"{outcome_icon} {info['outcome']:<6} | "
            f"{info['action']:<12} | "
            f"Stake: ${info['stake']:.2f} | "
            f"Reward: {info['reward']:+.1f} | "
            f"Bankroll: ${info['bankroll']:.2f}"
        )



#### 2.3 Mock Simulation to view the sample run 

In [6]:
# ----------------------------------------------------------------------
# Quick demo — run this file directly to see the environment in action
# ----------------------------------------------------------------------
if __name__ == "__main__":
    print("=" * 65)
    print("  Dragon Tiger Gym Environment — Random Agent Demo")
    print("=" * 65)

    env = DragonTigerEnv(
        num_decks=8,
        initial_bankroll=100.0,
        max_rounds=50,
        bet_size=1.0,
        render_mode="human",
    )

    obs, info = env.reset()
    print(f"\nStarting bankroll: ${info['bankroll']:.2f}\n")

    total_reward = 0.0
    step = 0
    round_stake = 5.0

    while True:
        action = env.action_space.sample()          # random agent
        obs, reward, terminated, truncated, info = env.step(action, stake=round_stake)
        total_reward += reward
        step += 1

        if terminated or truncated:
            break

    print("\n" + "=" * 65)
    print(f"  Game over after {step} rounds")
    print(f"  Final bankroll : ${info['bankroll']:.2f}")
    print(f"  Total reward   : {total_reward:+.1f}")
    print("=" * 65)

    env.close()

  Dragon Tiger Gym Environment — Random Agent Demo

Starting bankroll: $100.00

Round   1 | Dragon:  7  Tiger:  Q  🐯 Tiger  | Skip Round   | Stake: $0.00 | Reward: +0.0 | Bankroll: $100.00
Round   2 | Dragon:  J  Tiger:  7  🐉 Dragon | Bet Tiger    | Stake: $5.00 | Reward: -1.0 | Bankroll: $95.00
Round   3 | Dragon: 10  Tiger:  6  🐉 Dragon | Bet Tie      | Stake: $5.00 | Reward: -1.0 | Bankroll: $90.00
Round   4 | Dragon:  6  Tiger:  J  🐯 Tiger  | Skip Round   | Stake: $0.00 | Reward: +0.0 | Bankroll: $90.00
Round   5 | Dragon:  2  Tiger:  4  🐯 Tiger  | Bet Tiger    | Stake: $5.00 | Reward: +1.0 | Bankroll: $95.00
Round   6 | Dragon: 10  Tiger:  K  🐯 Tiger  | Walk Away    | Stake: $5.00 | Reward: -1.0 | Bankroll: $90.00

  Game over after 6 rounds
  Final bankroll : $90.00
  Total reward   : -2.0


____
### 3. Training the model (no walking away)

#### 3.1 Setting Hyper Parameters for training

In [7]:
# episodes = number of rounds the simulation will run 
EPISODES =100_000 

#alpha = amount to update the q-value each time, the learning rate of the model
ALPHA = 0.01  

#gamma = the amount to value future rewards 
GAMMA = 1.0       # no discounting — each round is self contained

#epsilon = exploration rate, eps = 1 means the agent starts at a completely random decision
EPSILON = 1.0

#minimum floor for epsilon kept at 0.01 so that there is a tiny bit of randomness left regardless
EPSILON_MIN = 0.01

#how much epsilon changes after each episode, kept to small amount so that decay is slow
EPSILON_DECAY = 0.999995

#### 3.2 Instantiating double Q-tables
- Create two Q-tables for Double Q-learning (keep 5 actions in table but avoid action 4 during training)
- For the purposes of this training run, skips the option of leaving the table

In [11]:
def create_qtable():
    return defaultdict(lambda: np.zeros(5))

q_table_a = create_qtable()
q_table_b = create_qtable()

In [12]:
# Discretise continuous observation into buckets for tabular Q-learning
NUM_BINS = 10
bins = [np.linspace(0.0, 1.0, NUM_BINS + 1)[1:-1] for _ in range(7)]
def discretize(obs):
    return tuple(int(np.digitize(obs[i], bins[i])) for i in range(len(obs)))
    # obs is length-7 vector in [0,1] — return a tuple of bin indices


def choose_action(state, eps): 
    if np.random.random() < eps: # random exploration limited to actions 0-3 (do not walk away during training)
        return np.random.randint(0, 4)
    combined = q_table_a[state] + q_table_b[state] #combine both tables but pick among first 4 actions only
    return int(np.argmax(combined[:4]))


def update_double_q(state, action, reward, next_state, done): #Double Q-learning update, randomly updating table A or B
    if np.random.random() < 0.5:
        best_next = int(np.argmax(q_table_a[next_state][:4]))
        target = reward + (0.0 if done else GAMMA * q_table_b[next_state][best_next])
        q_table_a[state][action] += ALPHA * (target - q_table_a[state][action])
    else:
        best_next = int(np.argmax(q_table_b[next_state][:4]))
        target = reward + (0.0 if done else GAMMA * q_table_a[next_state][best_next])
        q_table_b[state][action] += ALPHA * (target - q_table_b[state][action])


In [13]:
env = DragonTigerEnv(num_decks=8, initial_bankroll=100.0, max_rounds=50, bet_size=1.0, render_mode=None)
rewards = []
eps = EPSILON
for episode in range(EPISODES):
    obs, info = env.reset()
    state = discretize(obs)
    total_reward = 0.0
    done = False
    while not done:
        action = choose_action(state, eps)  # only 0-3 returned
        next_obs, reward, terminated, truncated, info = env.step(action, stake=1.0)
        done = terminated or truncated
        next_state = discretize(next_obs)

        update_double_q(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

    rewards.append(total_reward)
    eps = max(EPSILON_MIN, eps * EPSILON_DECAY)

    if (episode + 1) % 10000 == 0:
        avg = np.mean(rewards[-10000:])
        print(f"Episode {episode+1:>6} | Avg Reward (last 10k): {avg:.4f} | Epsilon: {eps:.4f}")

env.close()
print('Training complete')

Episode  10000 | Avg Reward (last 10k): -2.1372 | Epsilon: 0.9512
Episode  20000 | Avg Reward (last 10k): -2.1780 | Epsilon: 0.9048
Episode  30000 | Avg Reward (last 10k): -2.4475 | Epsilon: 0.8607
Episode  40000 | Avg Reward (last 10k): -2.5223 | Epsilon: 0.8187
Episode  50000 | Avg Reward (last 10k): -2.2792 | Epsilon: 0.7788
Episode  60000 | Avg Reward (last 10k): -2.0835 | Epsilon: 0.7408
Episode  70000 | Avg Reward (last 10k): -2.1599 | Epsilon: 0.7047
Episode  80000 | Avg Reward (last 10k): -2.0559 | Epsilon: 0.6703
Episode  90000 | Avg Reward (last 10k): -2.0005 | Epsilon: 0.6376
Episode 100000 | Avg Reward (last 10k): -1.9331 | Epsilon: 0.6065
Training complete


- evaluating the "Greedy" model that will not walk away 

In [15]:
# Simple evaluation: run greedy policy (still avoid action 4)
def evaluate(q_a, q_b, episodes=1000):
    env = DragonTigerEnv(num_decks=8, initial_bankroll=100.0, max_rounds=50, bet_size=1.0, render_mode=None)
    wins = losses = draws = 0
    for _ in range(episodes):
        obs, info = env.reset()
        state = discretize(obs)
        while True:
            combined = q_a[state] + q_b[state]
            action = int(np.argmax(combined[:4]))  # greedy among 0-3
            obs, reward, terminated, truncated, info = env.step(action, stake=1.0)
            state = discretize(obs)
            if terminated or truncated:
                if reward > 0:
                    wins += 1
                elif reward < 0:
                    losses += 1
                else:
                    draws += 1
                break
    env.close()
    total = wins + losses + draws
    print(f"Eval over {episodes} games — Wins: {wins}, Losses: {losses}, Draws: {draws}")
    return wins/total if total>0 else 0.0

win_rate = evaluate(q_table_a, q_table_b, episodes=2000)
print(f"Win rate (greedy, no walk-away): {win_rate:.3f}")

Eval over 2000 games — Wins: 393, Losses: 757, Draws: 850
Win rate (greedy, no walk-away): 0.197


____
### 4. Training a specialised model that reacts like a human (model is a class)

In [18]:
class DragonTigerAgent:
    def __init__(
        self,
        profit_target: float = 0.20,
        stop_loss: float = 0.20,
        loss_streak_limit: int = 5,
        alpha: float = 0.01,
        gamma: float = 1.0,
        epsilon: float = 1.0,
        eps_decay: float = 0.999995,
        eps_min: float = 0.01,
    ):
        self.profit_target = profit_target
        self.stop_loss = stop_loss
        self.loss_streak_limit = loss_streak_limit
        self.q_table_a = create_qtable()
        self.q_table_b = create_qtable()
        self.initial_bankroll = None
        self.loss_streak = 0
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.eps_decay = eps_decay
        self.eps_min = eps_min

    def reset(self, initial_bankroll: float):
        self.initial_bankroll = initial_bankroll
        self.loss_streak = 0

    def update_streak(self, reward: float):
        if reward < 0:
            self.loss_streak += 1
        elif reward > 0:
            self.loss_streak = 0

    def should_walk_away(self, bankroll: float) -> bool:
        if self.initial_bankroll is None:
            return False
        profit_hit = bankroll >= self.initial_bankroll * (1.0 + self.profit_target)
        stop_loss_hit = bankroll <= self.initial_bankroll * (1.0 - self.stop_loss)
        streak_hit = self.loss_streak >= self.loss_streak_limit
        return profit_hit or stop_loss_hit or streak_hit

    def choose_action(self, state, eps):
        if np.random.random() < eps:
            return np.random.randint(0, 4)
        combined = self.q_table_a[state] + self.q_table_b[state]
        return int(np.argmax(combined[:4]))

    def update(self, state, action, reward, next_state, done):
        if np.random.random() < 0.5:
            best_next = int(np.argmax(self.q_table_a[next_state][:4]))
            target = reward + (0.0 if done else self.gamma * self.q_table_b[next_state][best_next])
            self.q_table_a[state][action] += self.alpha * (target - self.q_table_a[state][action])
        else:
            best_next = int(np.argmax(self.q_table_b[next_state][:4]))
            target = reward + (0.0 if done else self.gamma * self.q_table_a[next_state][best_next])
            self.q_table_b[state][action] += self.alpha * (target - self.q_table_b[state][action])

    def act(self, obs, bankroll: float):
        if self.should_walk_away(bankroll):
            return 4
        state = discretize(obs)
        combined = self.q_table_a[state] + self.q_table_b[state]
        return int(np.argmax(combined[:4]))

    def train(
        self,
        env: DragonTigerEnv,
        episodes: int = 100_000,
        log_every: int = 10_000,
    ) -> list[float]:
        eps = self.epsilon
        rewards = []

        for episode in range(episodes):
            obs, info = env.reset()
            self.reset(initial_bankroll=info["bankroll"])
            state = discretize(obs)
            total_reward = 0.0
            done = False

            while not done:
                action = self.choose_action(state, eps)
                next_obs, reward, terminated, truncated, info = env.step(action, stake=1.0)
                done = terminated or truncated
                next_state = discretize(next_obs)

                self.update(state, action, reward, next_state, done)
                self.update_streak(reward)

                state = next_state
                total_reward += reward

            rewards.append(total_reward)
            eps = max(self.eps_min, eps * self.eps_decay)

            if (episode + 1) % log_every == 0:
                avg = np.mean(rewards[-log_every:])
                print(f"Episode {episode + 1:>6} | Avg Reward (last {log_every:,}): {avg:.4f} | Epsilon: {eps:.4f}")

        print("Training complete.")
        return rewards

    def evaluate(
        self,
        env: DragonTigerEnv,
        episodes: int = 2_000,
    ) -> dict:
        bankrolls = []
        walk_aways = 0
        busts = 0

        for _ in range(episodes):
            obs, info = env.reset()
            self.reset(initial_bankroll=info["bankroll"])

            while True:
                action = self.act(obs, bankroll=info["bankroll"])
                obs, reward, terminated, truncated, info = env.step(action, stake=1.0)
                self.update_streak(reward)

                if terminated or truncated:
                    bankrolls.append(info["bankroll"])
                    if action == 4:
                        walk_aways += 1
                    if info["bankroll"] <= 0:
                        busts += 1
                    break

        avg_bankroll = np.mean(bankrolls)
        profit_rate = sum(1 for b in bankrolls if b > env.initial_bankroll) / episodes

        print(f"\nEvaluation over {episodes:,} episodes")
        print(f"Avg final bankroll : ${avg_bankroll:.2f}")
        print(f"Profitable sessions: {profit_rate * 100:.1f}%")
        print(f"Walk-aways         : {walk_aways} ({walk_aways / episodes * 100:.1f}%)")
        print(f"Busts              : {busts} ({busts / episodes * 100:.1f}%)")

        return {
            "avg_bankroll": avg_bankroll,
            "profit_rate": profit_rate,
            "walk_aways": walk_aways,
            "busts": busts,
            "bankroll_history": bankrolls,
        }

In [19]:
env = DragonTigerEnv(num_decks=8, initial_bankroll=100.0, max_rounds=50)
agent = DragonTigerAgent(profit_target=0.20, stop_loss=0.20, loss_streak_limit=5)

rewards = agent.train(env, episodes=100_000)
results = agent.evaluate(env, episodes=2_000)

Episode  10000 | Avg Reward (last 10,000): -2.2285 | Epsilon: 0.9512
Episode  20000 | Avg Reward (last 10,000): -2.0412 | Epsilon: 0.9048
Episode  30000 | Avg Reward (last 10,000): -2.1778 | Epsilon: 0.8607
Episode  40000 | Avg Reward (last 10,000): -2.0538 | Epsilon: 0.8187
Episode  50000 | Avg Reward (last 10,000): -1.9812 | Epsilon: 0.7788
Episode  60000 | Avg Reward (last 10,000): -2.1772 | Epsilon: 0.7408
Episode  70000 | Avg Reward (last 10,000): -2.3009 | Epsilon: 0.7047
Episode  80000 | Avg Reward (last 10,000): -2.2571 | Epsilon: 0.6703
Episode  90000 | Avg Reward (last 10,000): -2.0758 | Epsilon: 0.6376
Episode 100000 | Avg Reward (last 10,000): -1.9221 | Epsilon: 0.6065
Training complete.

Evaluation over 2,000 episodes
Avg final bankroll : $98.63
Profitable sessions: 32.5%
Walk-aways         : 1642 (82.1%)
Busts              : 0 (0.0%)


- With burning 10 random cards at the start of the game, this game becomes increasingly unpredictable and impossible to win.
- A trained model, one that becomes slightly skewed on one side will likely incur a loss when evaluated
- However as the number of training episodes approach infinity, the model will approach a seemingly 50/50 decision between dragon and tiger
- This would stand a higher chance to profit.